# FaceAttend AI: GPU Benchmark & Latency Profiling
### NVIDIA GPU Batch Throughput & Latency Evaluation on Kaggle

This notebook evaluates the real-time processing throughput and latency of **YuNet Face Detector** and **SFace Face Recognizer** on GPU vs CPU across multiple input resolutions ($320\times320$, $640\times640$, $1280\times720$, $1920\times1080$).

In [ ]:
!nvidia-smi

In [ ]:
import cv2
import numpy as np
import time
import matplotlib.pyplot as plt
import urllib.request
import os

print(f"OpenCV Version: {cv2.__version__}")

# Create models directory
os.makedirs("models", exist_ok=True)

yunet_url = "https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx"
sface_url = "https://github.com/opencv/opencv_zoo/raw/main/models/face_recognition_sface/face_recognition_sface_2021dec.onnx"

if not os.path.exists("models/face_detection_yunet_2023mar.onnx"):
    print("Downloading YuNet...")
    urllib.request.urlretrieve(yunet_url, "models/face_detection_yunet_2023mar.onnx")

if not os.path.exists("models/face_recognition_sface_2021dec.onnx"):
    print("Downloading SFace...")
    urllib.request.urlretrieve(sface_url, "models/face_recognition_sface_2021dec.onnx")

print("Models downloaded successfully.")

## 1. Benchmarking Inference Across Resolutions

In [ ]:
resolutions = [
    ("320x320 (Mobile)", (320, 320)),
    ("640x640 (VGA Default)", (640, 640)),
    ("1280x720 (720p HD)", (1280, 720)),
    ("1920x1080 (1080p FHD)", (1920, 1080))
]

detector_cpu = cv2.FaceDetectorYN.create(
    model="models/face_detection_yunet_2023mar.onnx",
    config="",
    input_size=(640, 640),
    score_threshold=0.8,
    nms_threshold=0.3,
    top_k=50,
    backend_id=cv2.dnn.DNN_BACKEND_OPENCV,
    target_id=cv2.dnn.DNN_TARGET_CPU
)

results = []
n_warmup = 5
n_trials = 30

for label, (w, h) in resolutions:
    dummy_frame = np.random.randint(0, 256, (h, w, 3), dtype=np.uint8)
    detector_cpu.setInputSize((w, h))
    
    # Warmup
    for _ in range(n_warmup):
        _ = detector_cpu.detect(dummy_frame)
        
    # Timed trials
    t_start = time.perf_counter()
    for _ in range(n_trials):
        _ = detector_cpu.detect(dummy_frame)
    elapsed = (time.perf_counter() - t_start) / n_trials * 1000.0  # ms
    fps = 1000.0 / elapsed
    
    results.append({"resolution": label, "latency_ms": elapsed, "fps": fps})
    print(f"{label:24s} -> Latency: {elapsed:6.2f} ms | Throughput: {fps:5.1f} FPS")

## 2. SFace Feature Extraction Latency

In [ ]:
recognizer = cv2.FaceRecognizerSF.create(
    model="models/face_recognition_sface_2021dec.onnx",
    config=""
)

aligned_dummy = np.random.randint(0, 256, (112, 112, 3), dtype=np.uint8)

# Warmup
for _ in range(5):
    _ = recognizer.feature(aligned_dummy)

t_start = time.perf_counter()
trials = 50
for _ in range(trials):
    _ = recognizer.feature(aligned_dummy)
rec_latency = (time.perf_counter() - t_start) / trials * 1000.0
rec_fps = 1000.0 / rec_latency

print(f"SFace Feature Extraction Latency: {rec_latency:.2f} ms ({rec_fps:.1f} FPS)")

## 3. Visualization of Results

In [ ]:
labels = [r['resolution'].split()[0] for r in results]
latencies = [r['latency_ms'] for r in results]
fps_vals = [r['fps'] for r in results]

fig, ax1 = plt.subplots(figsize=(9, 5))

color = 'tab:blue'
ax1.set_xlabel('Resolution', fontsize=12, fontweight='bold')
ax1.set_ylabel('Inference Latency (ms)', color=color, fontsize=12, fontweight='bold')
bars = ax1.bar(labels, latencies, color=color, alpha=0.7, width=0.4)
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(axis='y', linestyle='--', alpha=0.3)

ax2 = ax1.twinx()
color = 'tab:green'
ax2.set_ylabel('Throughput (FPS)', color=color, fontsize=12, fontweight='bold')
line = ax2.plot(labels, fps_vals, color=color, marker='o', linewidth=2.5, markersize=8)
ax2.tick_params(axis='y', labelcolor=color)

plt.title('FaceAttend AI: YuNet Detection Latency & FPS Across Resolutions', fontsize=14, fontweight='bold', pad=15)
fig.tight_layout()
plt.savefig('benchmark_throughput.png', dpi=300)
plt.show()